In [1]:
!pip install google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 38.7 MB/s eta 0:00:00m eta 0:00:010:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.4
    Uninstalling protobuf-6.33.4:
      Successfully uninstalled protobuf-6.33.4
  Attempting uninstall: grpcio-status━━━━━━━━━━━━━━━━━ 0/7 [protobuf]
    Found existing installation: grpcio-status 1.76.0m 0/7 [protobuf]
    Uninstalling grpcio-status-1.76.0:━━━━━━━━━━━━ 0/7 [protobuf]
      Successfully uninstalled grpcio-status-1.76.0 0/7 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [google-generativeai]0m 5/7 [google

In [33]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    Tool
)

load_dotenv()
# 1. Configuración del Cliente (Uso de variables de entorno por seguridad)
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Definición del modelo y la configuración de búsqueda
MODEL_ID = "gemini-2.5-flash-lite"

def buscar_con_grounding(pregunta):
    try:
        # 3. Ejecución de la consulta con la herramienta de Google Search
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )

        # 4. Impresión del texto principal de la respuesta
        print(f"\n--- RESPUESTA ---\n{response.text}")

        # 5. Procesamiento detallado de fuentes (Grounding Metadata)
        metadata = response.candidates[0].grounding_metadata
        
        if metadata and metadata.search_entry_point:
            print("\n--- FUENTES VERIFICADAS ---")
            # Iteramos sobre los fragmentos (chunks) que contienen información de la web
            if metadata.grounding_chunks:
                for chunk in metadata.grounding_chunks:
                    if chunk.web:
                        print(f"• {chunk.web.title}")
                        print(f"  Enlace: {chunk.web.uri}")
        else:
            print("\nNo se utilizaron fuentes externas específicas.")

    except Exception as e:
        print(f"Error al conectar con la API: {e}")

# Bloque de ejecución principal
if __name__ == "__main__":
    pregunta_usuario = "¿Cuándo es el próximo eclipse solar total en España?"
    buscar_con_grounding(pregunta_usuario)


--- RESPUESTA ---
El próximo eclipse solar total visible en España será el 12 de agosto de 2026. Este fenómeno ocurrirá al atardecer y será visible en la mitad norte de la península ibérica, mientras que en la mitad sur se observará como parcial. La franja de totalidad cruzará el país de oeste a este, comenzando en Galicia y terminando en las Islas Baleares.

Después de este, se producirá otro eclipse solar total el 2 de agosto de 2027. Este eclipse será visible en algunas áreas específicas de España, como las ciudades autónomas de Ceuta y Melilla, la provincia de Cádiz, y el sur de las provincias de Granada y Almería. En otras partes de España, se verá como parcial.

Finalmente, el 26 de enero de 2028, España será testigo de un eclipse solar anular. Este evento, conocido como el "anillo de fuego", será visible en casi la mitad de España, de suroeste a noreste.

Es importante recordar que para observar cualquier tipo de eclipse solar de forma segura, se deben utilizar gafas especiales

In [36]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

# 1. Configuración del Cliente con API Key segura desde .env
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Definición del modelo
model = genai.GenerativeModel("gemini-2.5-flash-lite")

# 3. Pregunta sin grounding (búsqueda en Google desactivada)
pregunta = "¿Cuál fue el resultado del último partido del Real Madrid?"

# 4. Generar respuesta SIN herramientas de búsqueda
response = model.generate_content(pregunta)

# 5. Mostrar resultado
print("=== RESPUESTA SIN GROUNDING ===")
print(response.text)

# 6. Comparación: muestra si tiene metadatos de grounding
try:
    metadata = response.candidates[0].grounding_metadata
    if metadata and metadata.grounding_chunks:
        print("\n⚠️ Esta respuesta INCLUYE fuentes verificadas (Grounding activado)")
    else:
        print("\n⚠️ Esta respuesta se basa SOLO en el entrenamiento interno (SIN Grounding)")
except AttributeError:
    print("\n⚠️ Esta respuesta se basa SOLO en el entrenamiento interno (SIN Grounding)")

=== RESPUESTA SIN GROUNDING ===
Para darte el resultado del último partido del Real Madrid, necesito saber **cuándo** es "último" para ti.

Por favor, especifica la fecha o dime si te refieres al partido más reciente que se ha jugado.

⚠️ Esta respuesta se basa SOLO en el entrenamiento interno (SIN Grounding)


In [38]:
import time

# Solicitar al usuario que ingrese una pregunta o noticia de actualidad
pregunta_input = input("Introduce tu noticia o duda de actualidad: ")

MODEL_ID = "gemini-2.5-flash-lite"

# Configuración de reintentos para manejar indisponibilidad temporal del servicio
max_retries = 3  # Número máximo de intentos
retry_delay = 5  # Segundos de espera entre reintentos

# Bucle de reintentos: intenta conectar con la API hasta 3 veces
for attempt in range(max_retries):
    try:
        # Realizar la consulta a la API de Gemini con herramienta de Google Search
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta_input,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]  # Activar búsqueda en Google
            )
        )
        break  # Si tiene éxito, salir del bucle de reintentos
        
    except Exception as e:
        # Manejo de errores: reintentar si aún hay intentos disponibles
        if attempt < max_retries - 1:
            print(f"Intento {attempt + 1} fallido: {e}")
            print(f"Reintentando en {retry_delay} segundos...")
            time.sleep(retry_delay)  # Esperar antes de reintentar
        else:
            # Si se agotan los reintentos, lanzar el error
            print(f"Error después de {max_retries} intentos: {e}")
            raise

# Mostrar la respuesta generada y verificada por Google Search
print("\n=== RESPUESTA VERIFICADA ===")
print(response.text)

# Extraer y mostrar las fuentes utilizadas en la respuesta (Grounding)
try:
    # Obtener los fragmentos (chunks) con información de las fuentes web
    chunks = response.candidates[0].grounding_metadata.grounding_chunks
    
    if chunks:
        # Si hay fuentes, mostrarlas de forma organizada
        print("\n=== FUENTES ===")
        for i, chunk in enumerate(chunks, 1):
            # Extraer título y URL de cada fuente
            titulo = chunk.web.title
            url = chunk.web.uri
            print(f"{i}. {titulo}\n   {url}")
    else:
        # Si no hay fuentes específicas, mostrar advertencia
        print("\nAdvertencia: Esta respuesta se basa en mi entrenamiento "
              "interno y no ha sido verificada en tiempo real.")
              
except AttributeError:
    # Si no hay metadatos de grounding disponibles, mostrar advertencia
    print("\nAdvertencia: Esta respuesta se basa en mi entrenamiento "
          "interno y no ha sido verificada en tiempo real.")


=== RESPUESTA VERIFICADA ===
El último resultado del Real Madrid fue una derrota por 4-3 contra el Bayern de Múnich en la UEFA Champions League.

=== FUENTES ===
1. sofascore.com
   https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGIl0KI4qoWB6i1jNaIMc8P1LE1avBLkM1lc2A2tTrKMkA7ymeFg4OEAxr5An2r_2C0LaKSUwaf19MkAc4fZOr07sPCJFEYlad6M-y310-fOLMGLbT4KXpJzMy88FcSLxDpH9qhb356bTfMi6A_zMErPj6TwPh7
